In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [7]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [8]:
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [9]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000)

In [10]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [12]:
for epoch in range(3):
    model.train()

    for images, labels in train_loader:
        preds = model(images)
        loss = loss_fn(preds, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.2653
Epoch 2, Loss: 0.0634
Epoch 3, Loss: 0.0887


In [13]:
model.eval()

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=128, bias=True)
  (2): ReLU()
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Linear(in_features=64, out_features=10, bias=True)
)

In [14]:
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        correct += (model(images).argmax(1) == labels).sum().item()

print(f"准确率: {correct / len(test_set) * 100:.1f}%")

准确率: 97.5%


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ========== 新增GPU设备配置 ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 数据预处理：转 Tensor + 归一化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST 均值和标准差
])

# 加载 MNIST 数据集（自动下载）
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000)

# 定义网络：两个全连接隐藏层
model = nn.Sequential(
    nn.Flatten(),           # 28x28 → 784
    nn.Linear(784, 128),    # 第一层
    nn.ReLU(),
    nn.Linear(128, 64),     # 第二层
    nn.ReLU(),
    nn.Linear(64, 10)       # 输出层：10 个类别
).to(device)  # 模型移至GPU

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# 训练 3 个 epoch
for epoch in range(3):
    model.train()
    for images, labels in train_loader:
        # ========== 数据迁移到GPU ==========
        images, labels = images.to(device), labels.to(device)

        preds = model(images)          # 前向传播
        loss = loss_fn(preds, labels)  # 计算损失
        optimizer.zero_grad()          # 清零梯度
        loss.backward()                # 反向传播
        optimizer.step()               # 更新权重
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 测试准确率
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        correct += (outputs.argmax(1) == labels).sum().item()
print(f"准确率: {correct / len(test_set) * 100:.1f}%")

使用设备: cuda
Epoch 1, Loss: 0.0214
Epoch 2, Loss: 0.0165
Epoch 3, Loss: 0.1185
准确率: 97.4%
